# **Quantia Analytics Capstone Project**

## **Introduction**

> Heart disease remains one of the most critical public health challenges globally, accounting for millions of deaths every year. Early identification and intervention can significantly reduce complications and improve patient survival rates. However, due to limited healthcare resources and late diagnosis, many high-risk individuals often go unnoticed until the condition becomes severe.

> This project aims to build a Heart Disease Risk Prediction System that leverages patient health features to determine whether a person is at risk of developing heart disease.
> The study involves:

> 1. Conducting comprehensive Exploratory Data Analysis (EDA) to understand correlations and identify major contributing risk factors
> 2. Preprocessing and encoding the features for machine learning use
> 3. Applying and evaluating multiple classification algorithms
> 4. Selecting the best-performing model for real-world clinical use

### **Step 1: Exploratory Data Analysis**

#### Importing Relevant Libraries

In [106]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyparsing import col
import seaborn as sns
import joblib

from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

#  Scale Insensitive Models 
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Scale Sensitive Models 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression


from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc, recall_score, precision_score, f1_score
# from sklearn.utils import resample

# Colour Pallette for plots
color_list = ['#660000','#e69138','#990000','#f6b26b','#ead1dc']
color_list2 = ['#660000','#e69138','#990000','#f6b26b']
color = ['#e69138','#660000']


#### Loading the Dataset

In [107]:

data_path = Path(r"..\Healthcare\dataset\heart.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")


df = pd.read_csv(data_path)
print("Dataset loaded from:", data_path)
print("\nShape:", df.shape)

Dataset loaded from: ..\Healthcare\dataset\heart.csv

Shape: (1025, 14)


#### Creating a Directory for Analysis and Model output

In [108]:
# Creating an output directory for analysis visualizations
out_dir = Path(r"..\Healthcare\analysis_output")
out_dir.mkdir(exist_ok=True)

In [109]:
# Creating an output directory for Model
mod_out_dir = Path(r"..\Healthcare\modeloutput")
mod_out_dir.mkdir(exist_ok=True)

#### Assessing the dataset for Quality and Tidiness issues

In [110]:
#  Previewing the dataframe 
df.head ()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [111]:
# Checking for missing values
print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [112]:
# Checking for duplicated values
dupvalues = df.duplicated().any()
if dupvalues:
    no_of_dup = df.duplicated().sum()
    print(f"\nThere are {no_of_dup} duplicate rows in the dataset.")
else:
    print("\nNo duplicate rows found in the dataset.")


There are 723 duplicate rows in the dataset.


In [113]:
# Checking the metadata of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [114]:
# Checking the statistical description of the dataset
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.00000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,54.434146,0.695610,0.942439,131.611707,246.00000,0.149268,0.529756,149.114146,0.336585,1.071512,1.385366,0.754146,2.323902,0.513171
std,9.072290,0.460373,1.029641,17.516718,51.59251,0.356527,0.527878,23.005724,0.472772,1.175053,0.617755,1.030798,0.620660,0.500070
min,29.000000,0.000000,0.000000,94.000000,126.00000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.000000,0.000000,0.000000,120.000000,211.00000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,56.000000,1.000000,1.000000,130.000000,240.00000,0.000000,1.000000,152.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,275.00000,0.000000,1.000000,166.000000,1.000000,1.800000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.00000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


#### Exploring The Dataset 

In [115]:
# Identified our target column as "target"

print("\nTarget value counts:")
print(df["target"].value_counts(dropna=False))


Target value counts:
target
1    526
0    499
Name: count, dtype: int64


#### **Target Count Distribution**

In [116]:
sns.countplot(x="target", data=df, hue = 'target', palette = color)
plt.title('Heart Disease Class Distribution')
plt.xticks(ticks=[0, 1], labels=['No Disease', 'Heart Disease'])
plt.legend(title='Target', labels=['No Disease (0)', 'Heart Disease (1)'])
plt.xlabel('Target')
plt.ylabel('Count')
plt.savefig(out_dir / "Heart Disesase dist.png", bbox_inches='tight')
plt.close()
plt.show()

#### **Correlation matrix of the dataset**

In [117]:
# Depicting the correlation matrix of the dataset

plt.figure(figsize=(15,6))
plt.title("Correlation matrix Heat map")
sns.heatmap(df.corr(),annot=True)
plt.savefig(out_dir / "heatmap.png", bbox_inches='tight')
# plt.show()
plt.close()

#### **Insight**

> The correlation values show that chest pain type (cp) and maximum heart rate (thalach) have a moderate positive relationship with heart disease, meaning patients experiencing certain chest pain types or reaching higher maximum heart rates are more likely to have heart disease.
> Conversely, exercise-induced angina (exang) and ST depression (oldpeak) have moderate negative correlations with the target, indicating that those without exercise-induced chest pain and with lower ST depression levels are more likely to be classified as having heart disease.

# 5. Exploratory Analysis

### Visualizing Potential Outliers

In [118]:
features_to_check = ['age','trestbps', 'chol', 'thalach', 'oldpeak']
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[features_to_check], palette=color_list)
plt.title("Potential Outliers in Key Numeric Features")
plt.xticks(rotation=45)
plt.savefig(out_dir / "Potential Outliers in Key Numeric Features.png", bbox_inches='tight')
# plt.show()
plt.close()

### Visualising and Transforming highly skewed features 

In [119]:

selected_features = ['age', 'trestbps', 'chol', 'thalach']

# Check skewness only for these features
skew_vals = df[selected_features].skew().sort_values(ascending=False)
print("Feature Skewness in Selected Features:\n", skew_vals)

#  Identify features with high right skewness (> 1)
right_skewed = skew_vals[skew_vals > 1].index.tolist()
print("\nHighly Right-Skewed Features:", right_skewed)

#  Apply log1p (log(1 + x)) transformation only to those features
df_subset_log = df.copy()
df_subset_log[right_skewed] = np.log1p(df_subset_log[right_skewed])

#  Compare before vs after transformation visually
for col in right_skewed:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[col], kde=True, bins=20, color= color[0])
    plt.title(f"Before Log Transform: {col}")
    
    plt.subplot(1, 2, 2)
    sns.histplot(df_subset_log[col], kde=True, bins=20, color= color[1])
    plt.title(f"After Log Transform: {col}")
    
    plt.tight_layout()
    plt.savefig(out_dir / "Before and Fter Log Transform.png", bbox_inches='tight')
    # plt.show()
    plt.close()
    

Feature Skewness in Selected Features:
 chol        1.074073
trestbps    0.739768
age        -0.248866
thalach    -0.513777
dtype: float64

Highly Right-Skewed Features: ['chol']


### Capping the outliers

In [120]:
for col in ['trestbps', 'chol', 'thalach', 'oldpeak']:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)


### Visualising after Handling Outliers

In [121]:
sns.boxplot(data=df[['trestbps', 'chol', 'thalach', 'oldpeak']], palette=color_list2)
plt.title("After Outlier Handling")
plt.savefig(out_dir / "After Outlier Handling.png", bbox_inches='tight')
# plt.show()
plt.close()

### 1. Gender Distribution within Dataset

In [122]:
gender = df['sex'].value_counts()
labels = ['male', 'Female']
plt.figure(figsize=(6,6))
plt.pie(gender, labels=labels, colors=color, autopct='%1.1f%%', startangle=90)
plt.title("Gender Distribution")
plt.savefig(out_dir / "gender dist.png", bbox_inches='tight')
# plt.show()
plt.close()

### 2. Distribution of Heart Diseases among Males and Females 

In [123]:
sns.countplot(x='sex', hue='target', data=df,palette = color)
plt.xticks([0, 1], ['Female', 'Male'])
plt.title('Distribution of Heart Disease Among Gender')
plt.legend(title='Legend', labels=['No Heart diseases', 'with Heart diseases'], )
plt.xlabel('Gender')
plt.ylabel('Number of Patients')
plt.savefig(out_dir / "dist among gender.png", bbox_inches='tight')
# plt.show()
plt.close()

### 3. Distribution of Heart Diseases by Age 

In [124]:
#Depicting the distribution of heart disease by age groups
bins = [20, 30, 40, 50, 60, 70, 80]
df['age_range'] = pd.cut(df['age'], bins)

sns.countplot(x='age_range', hue='target', data=df, palette=color)
plt.title('Heart Disease Frequency by Age Group', fontsize=14)
plt.xlabel('Age Range')
plt.ylabel('Number of Patients')
plt.legend(title='Legend', labels=['No Heart diseases', 'with Heart diseases'], )
plt.savefig(out_dir / "Age Distribution by Heart Disease Status.png", bbox_inches='tight')
# plt.show()
plt.close()

In [125]:
sns.displot( df['age'],bins = 20, kde =True, color= color[1])
plt.title('Distribution of Age')
plt.ylabel('Number of Patients')
plt.xlabel('Age')
plt.savefig(out_dir / "dist by age.png", bbox_inches='tight')
# plt.show()
plt.close()

### 4. Distribution of Chest Pain Type Within Data set

In [126]:
#Showing the distribution of chest pain type 
sns.countplot(x='cp', data=df, palette=color_list2)    
plt.title('Distribution Of Chest Pain Tyoe  ')
plt.xticks([0, 1, 2, 3], ['typical angina', 'atypical angina', 'non-anginal pain', 'asymptomatic'])
plt.xlabel('Chest Pain Type')
plt.ylabel('Count')
plt.savefig(out_dir / "chest pain dist.png", bbox_inches='tight')
# plt.show()
plt.close()

C:\Users\Royal Technologies\AppData\Local\Temp\ipykernel_33480\486408017.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='cp', data=df, palette=color_list2)


### 5. Distribution of Heart Disease by Chest Pain

In [127]:
#Showing the distribution of heart disease by chest pain type
sns.countplot(data = df, x='cp', hue='target', palette=color)
plt.title('Chest Pain Type by Heart Disease')
plt.legend(title='Heart Disease', labels=['No Heart diseases', 'with Heart diseases'])
plt.xticks([0, 1, 2, 3], ['typical angina', 'atypical angina', 'non-anginal pain', 'asymptomatic'])
plt.xlabel('Chest Pain Type')
plt.ylabel('Count')
plt.savefig(out_dir / "dist by chest pain.png", bbox_inches='tight')
# plt.show()
plt.close()

### 6. Distribution of Heart Diseases by Blood Presure Levels

In [128]:
#Displaying the Distribution of Resting Blood Pressure
df['trestbps'].hist(edgecolor='black', color=color[0])
plt.title('Distribution of Resting Blood Pressure')
plt.savefig(out_dir / "bist by blood pressure.png", bbox_inches='tight')
# plt.show()
plt.close()

### 7. Distribution of Heart Diseases by Serum Cholesterol

In [129]:
#Displaying the Distribution of Cholesterol Levels
df['chol'].hist(edgecolor='black', color=color[0])
plt.title('Distribution of Cholesterol Levels')
plt.savefig(out_dir / "dist by cholesterol.png", bbox_inches='tight')
plt.xlabel('Cholesterol Levels')
plt.ylabel('Count')
# plt.show()
plt.close()


In [130]:
# Pairwise relationship between features
sns.pairplot(df[['age','chol','thalach','oldpeak','target']], hue='target', palette=color_list2)
plt.suptitle('Pairwise Feature Relationships', y=1.0)
plt.savefig(out_dir / "Pairwise Feature Relationships.png", bbox_inches='tight')
# plt.show()
plt.close()


c:\Users\Royal Technologies\UoPeople\.venv\Lib\site-packages\seaborn\axisgrid.py:1513: UserWarning: The palette list has more values (4) than needed (2), which may not be intended.
  func(x=vector, **plot_kwargs)
c:\Users\Royal Technologies\UoPeople\.venv\Lib\site-packages\seaborn\axisgrid.py:1513: UserWarning: The palette list has more values (4) than needed (2), which may not be intended.
  func(x=vector, **plot_kwargs)
c:\Users\Royal Technologies\UoPeople\.venv\Lib\site-packages\seaborn\axisgrid.py:1513: UserWarning: The palette list has more values (4) than needed (2), which may not be intended.
  func(x=vector, **plot_kwargs)
c:\Users\Royal Technologies\UoPeople\.venv\Lib\site-packages\seaborn\axisgrid.py:1513: UserWarning: The palette list has more values (4) than needed (2), which may not be intended.
  func(x=vector, **plot_kwargs)
c:\Users\Royal Technologies\UoPeople\.venv\Lib\site-packages\seaborn\axisgrid.py:1615: UserWarning: The palette list has more values (4) than needed

# 6. Training the Machine Learning Models

Spliting Data For Training 

In [131]:
x, y = df.drop('target', axis=1), df['target']

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y,test_size=0.2, random_state=9)
print("\nTrain shape:", x_train.shape, "Test shape:", x_test.shape)

# stratify=y ensures the same class ratio in both train and test sets.


Train shape: (820, 14) Test shape: (205, 14)


### Subsetting the data for required features 

In [132]:
# Choose a subset of important features
selected_features = ['age', 'cp','trestbps', 'chol', 'thalach' ]

# Create new training and test sets using only those features
x_train = x_train[selected_features]
x_test = x_test[selected_features]

## I. Train Test Split 

### Scalling for Scale Sensitive Models 

In [133]:
scaler = StandardScaler()
x_train_scaled= scaler.fit_transform(x_train)
x_test_scaled= scaler.transform(x_test)

## II. Training Scale Insensitive Models 

In [134]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [135]:
nb_clf = GaussianNB()
nb_clf.fit(x_train, y_train)

,priors,None
,var_smoothing,1e-09


In [136]:
gb_clf = GradientBoostingClassifier()
gb_clf.fit(x_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


## III. Training Scale Sensitive Models 

In [137]:
knn = KNeighborsClassifier()
knn.fit(x_train_scaled, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [138]:
log = LogisticRegression()
log.fit(x_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## IV. Evaluating Models 

### evaluating the accuracy, Precision, recall and F1-score for the models 

In [139]:
y_pred_rf = rf.predict(x_test)
y_pred_nb = nb_clf.predict(x_test)
y_pred_gb = gb_clf.predict(x_test)
y_pred_knn = knn.predict(x_test_scaled)
y_pred_log = log.predict(x_test_scaled)

In [140]:
# Random Forest metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

# Naive Bayes metrics
nb_clf_accuracy = accuracy_score(y_test, y_pred_nb)
nb_clf_precision = precision_score(y_test, y_pred_nb)
nb_clf_recall = recall_score(y_test, y_pred_nb)
nb_clf_f1 = f1_score(y_test, y_pred_nb)

# Gradient Boosting metrics
gb_clf_accuracy = accuracy_score(y_test, y_pred_gb)
gb_clf_precision = precision_score(y_test, y_pred_gb)
gb_clf_recall = recall_score(y_test, y_pred_gb)
gb_clf_f1 = f1_score(y_test, y_pred_gb)

# K-Nearest Neighbors metrics
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn)
knn_recall = recall_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn)

# Logistic Regression metrics
log_accuracy = accuracy_score(y_test, y_pred_log)
log_precision = precision_score(y_test, y_pred_log)
log_recall = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)

In [141]:
# Compile results into a DataFrame
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Naive Bayes",
        "Gradient Boosting",
        "K-Nearest Neighbors",
        "Logistic Regression",
        ],
    "Accuracy": [
        rf_accuracy, 
        nb_clf_accuracy,  
        gb_clf_accuracy,
        knn_accuracy, 
        log_accuracy
        ],
    "Precision": [
        rf_precision, 
        nb_clf_precision, 
        gb_clf_precision,
        knn_precision, 
        log_precision
        ],
    "Recall (Sensitivity)": [
        rf_recall, 
        nb_clf_recall,  
        gb_clf_recall, 
        knn_recall, 
        log_recall
        ],
    "F1-score": [
        rf_f1,
        nb_clf_recall,
        gb_clf_f1,
        knn_f1,
        log_f1
        ]
})

print("\nModel Performance Comparison:")
print(results.round(3))



Model Performance Comparison:
                 Model  Accuracy  Precision  Recall (Sensitivity)  F1-score
0        Random Forest     0.966      0.938                 1.000     0.968
1          Naive Bayes     0.727      0.709                 0.790     0.790
2    Gradient Boosting     0.893      0.888                 0.905     0.896
3  K-Nearest Neighbors     0.780      0.759                 0.838     0.796
4  Logistic Regression     0.693      0.675                 0.771     0.720


## V. Visualizing Area Under The Curve(A.U.C)

a. Random Forest

In [142]:
y_prob_rf = rf.predict_proba(x_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_rf)
plt.plot(fpr, tpr)
plt.title('Random Forest')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.savefig(mod_out_dir / "Random Forest.png", bbox_inches='tight')
plt.close()
# plt.show()
print('Area under the curve :', roc_auc_score(y_test, y_prob_rf))

Area under the curve : 1.0


b. Naive Bayes Classifier

In [143]:
y_prob_nb_clf = nb_clf.predict_proba(x_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_nb_clf)
plt.plot(fpr, tpr)
plt.title('Naive Bayes')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.savefig(mod_out_dir / "Naive Bayes.png", bbox_inches='tight')
plt.close()
# plt.show()
print('Area under the curve :', roc_auc_score(y_test, y_prob_nb_clf))

Area under the curve : 0.7882857142857143


c. Gradient Boosting Classifier

In [144]:
# y_prob = hgb_clf.predict_proba(x_test)[:, 1]
y_prob_gb_clf = gb_clf.predict_proba(x_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_gb_clf)
plt.plot(fpr, tpr)
plt.title('Gradiend Boosting classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.savefig(mod_out_dir / "Gradiend Boosting classifier.png", bbox_inches='tight')
plt.close()
# plt.show()
print('Area under the curve :', roc_auc_score(y_test, y_prob_gb_clf))

Area under the curve : 0.9722857142857143


d. K Nearest Neighbour

In [145]:
y_prob_knn = knn.predict_proba(x_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_knn)
plt.plot(fpr, tpr)
plt.title('K Nearest Neighbours')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.savefig(mod_out_dir / "K Nearest Neighbours.png", bbox_inches='tight')
plt.close()
# plt.show()
print('Area under the curve :', roc_auc_score(y_test, y_prob_knn))

Area under the curve : 0.9064285714285714


e. Logistic Regression

In [146]:
y_prob_log = log.predict_proba(x_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_log)
plt.plot(fpr, tpr)
plt.title('Logistic Regression')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
# plt.show()
plt.savefig(mod_out_dir / "Logistic Regression.png", bbox_inches='tight')
plt.close()
print('Area under the curve :', roc_auc_score(y_test, y_prob_log))

Area under the curve : 0.7792380952380952


## VI. Hyperparameter Tuning

In [147]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],        # number of trees
    'max_depth': [5, 10, 15, None],               # depth of each tree
    'min_samples_split': [2, 5, 10],        # minimum samples to split a node
    'min_samples_leaf': [1, 2, 4],          # minimum samples per leaf
    'max_features': ['sqrt', 'log2'],       # number of features to consider
    'bootstrap': [True, False]              # whether to bootstrap samples
}

# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='recall',     
    n_jobs=-1,           # use all CPU cores
    verbose=2
)

grid_search.fit(x_train, y_train)

Fitting 3 folds for each of 432 candidates, totalling 1296 fits


,estimator,RandomForestClassifier()
,param_grid,"{'bootstrap': [True, False], 'max_depth': [5, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [148]:
best_rf = grid_search.best_estimator_

In [149]:
best_rf

,n_estimators,200
,criterion,'gini'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [150]:
best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(x_test)
y_prob_rf= best_rf.predict_proba(x_test)[:, 1]

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
print("\nConfusion Matrix:\n", cm)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob_rf):.3f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Tuned Logistic Regression")
plt.savefig(mod_out_dir / "ROC Curve - Tuned Logistic Regression.png", bbox_inches='tight')
# plt.show()
plt.close()


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.96       100
           1       0.94      1.00      0.97       105

    accuracy                           0.97       205
   macro avg       0.97      0.97      0.97       205
weighted avg       0.97      0.97      0.97       205


Confusion Matrix:
 [[ 93   7]
 [  0 105]]


## VII. Feature Importances

In [151]:
# Get feature importance from tuned Random Forest Clasifier
rf_importance = pd.DataFrame({
    'Feature': x_train.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)


plt.figure(figsize=(8, 5))
plt.barh(rf_importance['Feature'], rf_importance['Importance'], color=color_list2)
plt.gca().invert_yaxis()  # highest importance at top
plt.title('Feature Importance - Random Forest Model')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.savefig(mod_out_dir / "Feature Importance.png", bbox_inches='tight')
# plt.show()
plt.close()


In [152]:
plt.figure(figsize=(12, 6))
sns.heatmap(abs(df.corr()), annot=True , fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.savefig(mod_out_dir / "Feature Correlation Heatmap.png", bbox_inches='tight')
# plt.show()
plt.close()


TypeError: float() argument must be a string or a real number, not 'pandas._libs.interval.Interval'

<Figure size 1200x600 with 0 Axes>

## Saving Model To File 

In [ ]:
model_path =mod_out_dir/ "best_random_forest_model.pkl"
joblib.dump((rf, selected_features), model_path)

print(f" Model saved successfully at: {model_path}")

 Model saved successfully at: ..\Healthcare\modeloutput\best_random_forest_model.pkl


## Loading the model from saved file 

In [ ]:
# Load the saved model
loaded_model, selected_features = joblib.load(model_path)

print(" Model loaded and ready for prediction.")

 Model loaded and ready for prediction.


## Using The Model With New Data

In [ ]:
new_patient = pd.DataFrame({
    'age': [60],
    'sex': [1],
    'cp': [2],
    'trestbps': [140],
    'chol': [250],
    'fbs': [0],
    'restecg': [1],
    'thalach': [140],
    'exang': [0],
    'oldpeak': [1.5],
    'slope': [2],
    'ca': [0],
    'thal': [2]
})

In [ ]:
# Predict with loaded model
# Use the same features for prediction
new_patient = df[selected_features]
prediction = loaded_model.predict(new_patient)[0]
print("Prediction:", "At Risk Of Heart Disease" if prediction == 1 else "Not at Risk of Heart Disease")


Prediction: Not at Risk of Heart Disease
